[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Many to Many &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup: the six classes, `hero_engine`, and `scratch/heroes.db` with
the eight heroes and three teams loaded. The missions and operations are made by the tasks, so run
them in order, and the last cell removes the scratch folder.


In [1]:
import re
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from sqlalchemy import event, insert, text
from sqlalchemy.dialects import sqlite
from sqlalchemy.exc import IntegrityError
from sqlalchemy.schema import CreateTable
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine, select

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


def table_sql(model):
    """The CREATE TABLE a table model describes, written for SQLite with no database anywhere."""
    return str(CreateTable(model.__table__).compile(dialect=sqlite.dialect())).strip()

def run_python(path):
    """Run a file in a Python of its own and print what it printed."""
    done = subprocess.run([sys.executable, path], capture_output=True, text=True)
    print(done.stdout.strip() or done.stderr.strip().splitlines()[-1])


class HeroMissionLink(SQLModel, table=True):
    """The table in the middle, with a key made of both sides."""

    hero_id: int | None = Field(default=None, foreign_key="hero.id", primary_key=True)
    mission_id: int | None = Field(default=None, foreign_key="mission.id", primary_key=True)
    role: str | None = Field(default=None, max_length=30)           # nothing can set this through the lists


class Deployment(SQLModel, table=True):
    """The same shape, written as a model with two attributes of its own."""

    hero_id: int | None = Field(default=None, foreign_key="hero.id", primary_key=True)
    operation_id: int | None = Field(default=None, foreign_key="operation.id", primary_key=True)
    role: str = Field(max_length=30)

    hero: "Hero" = Relationship(back_populates="deployments")
    operation: "Operation" = Relationship(back_populates="deployments")


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)

    heroes: list["Hero"] = Relationship(back_populates="team")


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")

    team: Team | None = Relationship(back_populates="heroes")
    missions: list["Mission"] = Relationship(back_populates="heroes", link_model=HeroMissionLink)
    deployments: list[Deployment] = Relationship(back_populates="hero")


class Mission(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    title: str = Field(unique=True, max_length=80)

    heroes: list[Hero] = Relationship(back_populates="missions", link_model=HeroMissionLink)


class Operation(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    code_name: str = Field(unique=True, max_length=80)

    deployments: list[Deployment] = Relationship(back_populates="operation")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)

with Session(engine) as session:
    print("sqlmodel", sqlmodel.__version__, "|", len(session.exec(select(Hero)).all()), "heroes |",
          "tables:", sorted(SQLModel.metadata.tables))


sqlmodel 0.0.42 | 8 heroes | tables: ['deployment', 'hero', 'heromissionlink', 'mission', 'operation', 'team']


**1.** A mission, and the rows it writes in the middle.


In [2]:
with Session(engine) as session:
    going = [session.get(Hero, hero_id) for hero_id in (1, 3, 5)]   # looked up first
    vault = Mission(title="Guard the vault")
    for hero in going:
        vault.heroes.append(hero)
    session.add(vault)
    session.commit()

    print("heroes   :", sorted(hero.name for hero in vault.heroes))
    print("link rows:", session.connection().exec_driver_sql(
        "select hero_id, mission_id, role from heromissionlink").fetchall())


heroes   : ['Black Lion', 'Deadpond', 'Rusty-Man']
link rows: [(3, 1, None), (5, 1, None), (1, 1, None)]


Three appends, three rows, and a role of `None` in each, since nothing in the list can set it.


**2.** Who is on a mission, and how many.


In [3]:
with Session(engine) as session:
    for hero in session.exec(select(Hero).order_by(Hero.name)):
        if hero.missions:
            print(f"  {hero.name:<22} {len(hero.missions)}")


  Black Lion             1
  Deadpond               1
  Rusty-Man              1


Reading `hero.missions` for every hero sends one query for each of them, which is the
**Loading and N+1** notebook's subject.


**3.** A hero taken off a mission.


In [4]:
with Session(engine) as session:
    vault = session.exec(select(Mission).where(Mission.title == "Guard the vault")).one()
    going = session.get(Hero, 5)
    vault.heroes.remove(going)
    session.commit()

    print("heroes now:", sorted(hero.name for hero in vault.heroes))
    print("link rows :", session.connection().exec_driver_sql(
        "select hero_id, mission_id from heromissionlink where mission_id = :id",
        {"id": vault.id}).fetchall())


heroes now: ['Deadpond', 'Rusty-Man']
link rows : [(1, 1), (3, 1)]


`remove` from the list deleted the row in the middle. The hero is untouched: it is the pairing that
ended.


**4.** An operation, with roles.


In [5]:
with Session(engine) as session:
    scout, driver = session.get(Hero, 2), session.get(Hero, 4)      # looked up first
    daybreak = Operation(code_name="Daybreak")
    daybreak.deployments.append(Deployment(hero=scout, role="scout"))
    daybreak.deployments.append(Deployment(hero=driver, role="driver"))
    session.add(daybreak)
    session.commit()

    for deployment in session.get(Operation, 1).deployments:
        print(f"  {deployment.hero.name} ({deployment.role})")


  Spider-Boy (scout)
  Tarantula (driver)


The role went in where the pairing was made, because the pairing is an object with a place to put
it.


**5.** Every role a hero has had.


In [6]:
with Session(engine) as session:
    for hero in session.exec(select(Hero).order_by(Hero.name)):
        if hero.deployments:
            print(f"  {hero.name}: {', '.join(sorted(d.role for d in hero.deployments))}")


  Spider-Boy: scout
  Tarantula: driver


From the hero's side the link rows are `hero.deployments`, and each one holds its own role.


**6.** Everything one hero is on.


In [7]:
def missions_of(session, hero_name):
    """The titles of a hero's missions and the code names of their operations."""
    hero = session.exec(select(Hero).where(Hero.name == hero_name)).one()
    return {"hero": hero.name,
            "missions": sorted(mission.title for mission in hero.missions),
            "operations": sorted(deployment.operation.code_name for deployment in hero.deployments)}


with Session(engine) as session:
    for name in ("Deadpond", "Spider-Boy", "Princess Sure-E"):
        print(missions_of(session, name))


{'hero': 'Deadpond', 'missions': ['Guard the vault'], 'operations': []}
{'hero': 'Spider-Boy', 'missions': [], 'operations': ['Daybreak']}
{'hero': 'Princess Sure-E', 'missions': [], 'operations': []}


One hero is on a mission and no operation, one on both, and one on neither, which comes back as two
empty lists rather than as an error.

Last, the engine lets go of the file, and this cell removes the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Many to Many](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/09-many-to-many.ipynb)  &nbsp;&middot;&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
